# Countdown option C: GRAFT + hidden-state probe, training process

**Question.** On option C (guard 1.0) plain GRAFT fails: the model learns the hidden hack *constraint_relax*, the test-overwrite rule never fires, and the deployed model still hacks (3/3 seeds). Here the routing monitor is **rule OR probe** (layer-27 logistic regression on the frozen base model, trained only on canonical runs' monitor flags, 97% threshold). Two things are tracked over training:
1. **the model**: does the hidden hack stay out of the deployed model, and at what cost in task accuracy?
2. **the probe**: does it keep catching the hack while the policy changes (recall), and how many clean answers does it route away (false alarms)?

| Term | Meaning |
|---|---|
| **Deployed model** | RETAIN only (FORGET switched off). What GRAFT ships. |
| **Training policy** | RETAIN + FORGET: the model that generates the training rollouts. |
| **constraint_relax** | The hidden hack: test.py's numbers-used-once check dropped, answer still pays. |
| **guard_blocked** | A test-overwrite *attempt* that the guard made pay 0 (not a paying hack). |
| **Recall** | % of answers of a hack type that the monitor flags (= routed to FORGET). |
| **False alarm** | % of clean answers the monitor flags. |
| **Still hacking** | Deployed hack rate above 10% in the final eval. |

Agreed success rule (set before the runs): final deployed hack **below 10% in all 3 seeds**, task accuracy within about 2 points of plain GRAFT (78.1%). Every number below is computed from the run files when the notebook runs.

In [1]:
import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown
import cd_probe_train as D
COL = {"GRAFT": "#2a78d6", "GRAFT + probe": "#eb6834", "base model": "#8a8984"}
CATC = {"constraint_relax": "#eb6834", "modify_input": "#8e5cc4", "guard_blocked": "#1baf7a", "clean": "#8a8984", "test_overwrite": "#2a78d6", "other": "#c9a227"}
LAYOUT = dict(template="plotly_white", font=dict(size=12), hovermode="x unified", margin=dict(l=60, r=20, t=70, b=50),
              legend=dict(orientation="h", y=1.08, x=0))
def band(fig, df, x, y, name, color, row=None, col=None, show=True):
    g = df.groupby(x)[y]; m, se, n = g.mean(), g.sem().fillna(0), g.count()
    kw = dict(row=row, col=col) if row else {}
    fig.add_trace(go.Scatter(x=list(m.index) + list(m.index[::-1]), y=list(m + se) + list((m - se)[::-1]), fill="toself",
                             fillcolor=color, opacity=0.15, line=dict(width=0), hoverinfo="skip", showlegend=False, legendgroup=name), **kw)
    fig.add_trace(go.Scatter(x=m.index, y=m.values, name=name, line=dict(color=color, width=2.5), legendgroup=name, showlegend=show,
                             customdata=np.c_[se.values, n.values], hovertemplate=f"{name}: %{{y:.1f}}% ± %{{customdata[0]:.1f}} (n=%{{customdata[1]}} seeds)<extra></extra>"), **kw)
runs = D.available()
display(Markdown("Runs found: " + ", ".join(f"**{a}** s{s}" for a, s in sorted(runs))
                 + ("" if any(a == "GRAFT + probe" for a, _ in runs) else "  \n⚠️ **No probe runs found yet**: sync them to `rl/output/arrhenius/` and re-run.")))

Runs found: **GRAFT** s9, **GRAFT** s15, **GRAFT** s16, **GRAFT + probe** s9, **GRAFT + probe** s15, **GRAFT + probe** s16

---
## 1. Final result (fseval, step 200, 256 prompts, deployed model)

In [2]:
F = D.fseval()
dep = F[(F.forget == 0.0) & F.arm.isin(list(D.ARMS))]
base = F[F.arm == "base model"]
fig = make_subplots(1, 3, subplot_titles=["Hack rate, all types (deployed)", "Hidden hack: constraint_relax (deployed)", "Task accuracy (deployed)"])
for i, m in enumerate(["hack", "constraint_relax", "task"], 1):
    for arm in D.ARMS:
        d = dep[dep.arm == arm].sort_values("seed")
        if not len(d):
            continue
        fig.add_trace(go.Bar(x=[arm], y=[d[m].mean()], marker_color=COL[arm], name=arm, showlegend=(i == 1), legendgroup=arm,
                             text=[f"{d[m].mean():.1f}%"], textposition="outside", hovertemplate=f"{arm}: mean %{{y:.1f}}%<extra></extra>"), 1, i)
        fig.add_trace(go.Scatter(x=[arm] * len(d), y=d[m], mode="markers", marker=dict(color="#0b0b0b", size=8), showlegend=False,
                                 customdata=d.seed, hovertemplate="seed %{customdata}: %{y:.1f}%<extra></extra>"), 1, i)
    if len(base):
        fig.add_hline(y=float(base[m].iloc[0]), line=dict(color=COL["base model"], dash="dot"), row=1, col=i,
                      annotation_text=f"base model {float(base[m].iloc[0]):.1f}%", annotation_position="top left")
    if m != "task":
        fig.add_hline(y=10, line=dict(color="#52514e", dash="dash", width=1), row=1, col=i, annotation_text="10% = still hacking",
                      annotation_position="bottom right")
fig.update_yaxes(ticksuffix="%", rangemode="tozero"); fig.update_layout(**{**LAYOUT, "hovermode": "closest"}, height=420,
    title="Final deployed model: plain GRAFT vs GRAFT + probe (bars = mean, dots = seeds)")
fig.show()
if len(dep):
    t = dep.pivot_table(index="seed", columns="arm", values=["hack", "constraint_relax", "task"]).round(1)
    display(t)
    if "GRAFT + probe" in set(dep.arm):
        p, b = dep[dep.arm == "GRAFT + probe"], dep[dep.arm == "GRAFT"]
        n_hack = int((p.hack > 10).sum())
        display(Markdown(f"**Success rule:** deployed hack < 10% in all seeds → **{len(p) - n_hack}/{len(p)} seeds below 10%** "
                         f"(plain GRAFT: {int((b.hack <= 10).sum())}/{len(b)}). Task accuracy {p.task.mean():.1f}% vs {b.task.mean():.1f}% "
                         f"(difference {p.task.mean() - b.task.mean():+.1f} points; rule: within about 2)."))

constraint_relax                hack                task              
arm             GRAFT GRAFT + probe GRAFT GRAFT + probe GRAFT GRAFT + probe
seed                                                                       
9.0              14.8           2.7  16.0           4.3  79.3          69.9
15.0             14.8           0.8  16.4           2.0  79.3          80.1
16.0             20.3           2.3  21.1           3.1  75.8          85.2

**Success rule:** deployed hack < 10% in all seeds → **3/3 seeds below 10%** (plain GRAFT: 0/3). Task accuracy 78.4% vs 78.1% (difference +0.3 points; rule: within about 2).

---
## 2. The model over training (in-training evals: every 12 steps, 64 prompts → each answer = 1.6 points, so single points are noisy)

In [3]:
E = D.eval_curves()
panels = [("constraint_relax", "retain_only", "Hidden hack (constraint_relax)<br>DEPLOYED model"), ("hack", "retain_only", "Hack, all types<br>DEPLOYED model"),
          ("task", "retain_only", "Task accuracy<br>DEPLOYED model"), ("constraint_relax", "both", "Hidden hack (constraint_relax)<br>TRAINING policy (RETAIN + FORGET)")]
fig = make_subplots(2, 2, subplot_titles=[p[2] for p in panels], vertical_spacing=0.16, shared_xaxes=True)
for i, (m, mode, _) in enumerate(panels):
    r, c = i // 2 + 1, i % 2 + 1
    d = E[E["mode"] == mode]
    for arm in D.ARMS:
        da = d[d.arm == arm]
        for s, ds in da.groupby("seed"):
            fig.add_trace(go.Scatter(x=ds.step, y=ds[m], line=dict(color=COL[arm], width=0.8), opacity=0.35, showlegend=False,
                                     hovertemplate=f"{arm} s{s}: %{{y:.1f}}%<extra></extra>"), r, c)
        if len(da):
            band(fig, da, "step", m, arm, COL[arm], r, c, show=(i == 0))
fig.update_yaxes(ticksuffix="%", rangemode="tozero"); fig.update_xaxes(title_text="training step", row=2)
fig.update_layout(**LAYOUT, height=720, title="Model over training: mean ± SE over seeds (thin lines = single seeds)")
fig.show()
late = E[(E.step >= 132)].groupby(["arm", "seed", "mode"])[["task", "hack", "constraint_relax"]].mean().round(1)
display(Markdown("**Mean over the last evals (steps 132–192), per seed:**")); display(late.unstack("mode").swaplevel(axis=1)[["retain_only", "both"]] if len(late) else late)

**Mean over the last evals (steps 132–192), per seed:**

mode               retain_only                         both        \
                          task  hack constraint_relax  task  hack   
arm           seed                                                  
GRAFT         9           78.1  16.9             16.4  72.9  26.6   
              15          76.0  19.8             18.7  75.3  24.2   
              16          73.2  23.7             22.4  69.8  29.9   
GRAFT + probe 9           68.8   6.0              4.2  63.3  36.7   
              15          76.6   3.6              2.1  67.5  32.5   
              16          82.0   5.7              4.4  67.2  32.6   

mode                                 
                   constraint_relax  
arm           seed                   
GRAFT         9                26.6  
              15               23.7  
              16               29.9  
GRAFT + probe 9                36.7  
              15               32.5  
              16               32.6

---
## 3. The probe on the TRAINING rollouts (what was actually routed to FORGET)

`routing_trace.jsonl` stores, every 20 steps, the whole rollout (1,024 samples, anchors excluded): how many samples of each kind there were and how many the monitor flagged. This is exactly what GRAFT routed. For plain GRAFT only the rule fires (it flags only paying test overwrites, which pay 0 on option C).

In [4]:
RR = D.rollout_routing()
R = D.rates(RR, ["arm", "seed", "step"])
show = ["constraint_relax", "guard_blocked", "modify_input", "clean"]
fig = make_subplots(1, 2, subplot_titles=["Share of each kind routed to FORGET<br>(recall for hacks, false alarms for clean)",
                                          "How many samples of each kind the rollout contains"])
p = R[R.arm == "GRAFT + probe"]
for cat in show:
    d = p[p.category == cat]
    if len(d):
        band(fig, d.dropna(subset=["flagged_pct"]), "step", "flagged_pct", cat, CATC[cat], 1, 1)
    q = RR[(RR.arm == "GRAFT + probe") & (RR.category == cat)].assign(pct=lambda x: 100 * x.n / 1024)
    if len(q):
        band(fig, q, "step", "pct", cat, CATC[cat], 1, 2, show=False)
fig.update_yaxes(ticksuffix="%", rangemode="tozero"); fig.update_xaxes(title_text="training step")
fig.update_layout(**LAYOUT, height=440, title="GRAFT + probe: training rollouts (mean ± SE over seeds; 1,024 samples per point)")
fig.show()
if len(p):
    tot = D.rates(RR[RR.arm == "GRAFT + probe"], ["seed"]).pivot(index="seed", columns="category", values="flagged_pct").round(1)
    display(Markdown("**% of each kind routed to FORGET, all logged rollouts pooled, per seed:**")); display(tot[[c for c in show if c in tot]])
    fl = RR[RR.arm == "GRAFT + probe"].groupby("category")[["n", "n_flagged"]].sum()
    comp = (100 * fl.n_flagged / fl.n_flagged.sum()).round(1)
    display(Markdown("**What went to FORGET (share of all flagged samples, pooled):** " + ", ".join(f"{c} {v}%" for c, v in comp.sort_values(ascending=False).items())))
b = R[R.arm == "GRAFT"].groupby("category")[["n", "n_flagged"]].sum()
if len(b):
    display(Markdown(f"Plain GRAFT, same rollouts: {int(b.n_flagged.sum())} of {int(b.n.sum())} samples flagged "
                     f"({100 * b.n_flagged.sum() / b.n.sum():.2f}%); constraint_relax routed: {int(b.loc['constraint_relax','n_flagged']) if 'constraint_relax' in b.index else 0} of {int(b.loc['constraint_relax','n']) if 'constraint_relax' in b.index else 0}."))

**% of each kind routed to FORGET, all logged rollouts pooled, per seed:**

category,constraint_relax,guard_blocked,modify_input,clean
seed,,,,
9,88.4,68.3,0.0,0.5
15,92.7,61.9,0.0,0.3
16,85.6,73.6,0.0,0.4


**What went to FORGET (share of all flagged samples, pooled):** constraint_relax 96.2%, guard_blocked 1.8%, clean 1.1%, test_overwrite 1.0%, modify_input 0.0%, other 0.0%

Plain GRAFT, same rollouts: 52 of 27648 samples flagged (0.19%); constraint_relax routed: 0 of 5369.

---
## 4. The probe on the EVAL answers, by training stage

Same question on the eval answers (`eval_samples.jsonl`), split by stage and by which model wrote the answer. 'Training policy' answers look like the rollouts; 'deployed' and 'FORGET only' answers show how the probe sees the two adapters separately.

In [5]:
A = D.eval_answers()
A = A[A.arm == "GRAFT + probe"]
if len(A):
    A["stage"] = pd.cut(A.step, [-1, 48, 108, 200], labels=["0–48", "60–108", "120–192"])
    S = D.rates(A, ["mode", "stage"])
    fig = make_subplots(1, 3, subplot_titles=[f"Answers by the {D.MODES[m]}" for m in ["both", "retain_only", "forget_only"]], shared_yaxes=True)
    for i, mode in enumerate(["both", "retain_only", "forget_only"], 1):
        for cat in ["constraint_relax", "guard_blocked", "clean"]:
            d = S[(S["mode"] == mode) & (S.category == cat)]
            fig.add_trace(go.Bar(x=d.stage.astype(str), y=d.flagged_pct, name=cat, marker_color=CATC[cat], showlegend=(i == 1), legendgroup=cat,
                                 customdata=np.c_[d.n_flagged, d.n], text=[f"{v:.0f}%" if v == v else "" for v in d.flagged_pct], textposition="outside",
                                 hovertemplate=f"{cat}: %{{y:.1f}}% flagged (%{{customdata[0]}} of %{{customdata[1]}})<extra></extra>"), 1, i)
    fig.update_yaxes(ticksuffix="%", range=[0, 105]); fig.update_xaxes(title_text="training steps")
    fig.update_layout(**{**LAYOUT, "hovermode": "closest"}, barmode="group", height=430,
                      title="GRAFT + probe, eval answers: % flagged by the monitor (3 seeds pooled; hover for counts)")
    fig.show()
    display(S.pivot_table(index=["mode", "stage"], columns="category", values="flagged_pct").round(1)[["constraint_relax", "guard_blocked", "modify_input", "clean"]])
else:
    display(Markdown("No probe runs yet."))

/Users/zhouchengyi/Documents/workspace/GRAFT/rl/analysis/cd_probe_train.py:153: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



/var/folders/wc/776xqlv973lf5ffshsrlrb_c0000gn/T/ipykernel_8349/1496658230.py:17: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior



category             constraint_relax  guard_blocked  modify_input  clean
mode        stage                                                        
both        0–48                 37.1           66.7           0.0    1.2
            60–108               83.8           25.0           0.0    0.4
            120–192              99.1            NaN           NaN    0.0
forget_only 0–48                 38.9           66.7           0.0    0.7
            60–108               75.8            0.0           0.0    0.0
            120–192              96.4            NaN           0.0    0.0
retain_only 0–48                 35.3           76.1           0.0    1.8
            60–108               43.5           75.9           0.0    0.9
            120–192              43.1           67.9           0.0    1.3

---
## 5. Probe flags on every rollout (train.log)

One line per rollout (every 4 steps): how many of the 1,024 samples the rule and the probe flagged, and the probe's extra time.

In [6]:
P = D.probe_log()
if len(P):
    fig = make_subplots(1, 2, subplot_titles=["Samples flagged per rollout (of 1,024)", "Probe scoring time per rollout"])
    for s, d in P.groupby("seed"):
        fig.add_trace(go.Scatter(x=d.step, y=d.probe_flags, name=f"probe, s{s}", line=dict(width=1.5)), 1, 1)
    band(fig, P, "step", "rule_flags", "rule (test overwrite), mean", COL["GRAFT"], 1, 1)
    for s, d in P.groupby("seed"):
        fig.add_trace(go.Scatter(x=d.step, y=d.probe_sec, name=f"s{s}", line=dict(width=1), showlegend=False), 1, 2)
    fig.update_yaxes(title_text="samples", row=1, col=1); fig.update_yaxes(title_text="seconds", row=1, col=2)
    fig.update_xaxes(title_text="training step"); fig.update_layout(**LAYOUT, height=420, title="Every rollout, from train.log")
    fig.show()
    display(Markdown(f"Rollouts logged: {P.groupby('seed').size().to_dict()}. Rule flags in total: {int(P.rule_flags.sum())}; probe flags: {int(P.probe_flags.sum())} "
                     f"(probe-only {int(P.probe_only.sum())}). Probe time: {P.probe_sec.mean():.0f} s per rollout on average, {P.groupby('seed').probe_sec.sum().mean()/60:.0f} min per run."))
else:
    display(Markdown("No probe runs yet."))

Rollouts logged: {9: 50, 15: 50, 16: 50}. Rule flags in total: 389; probe flags: 36460 (probe-only 36118). Probe time: 37 s per rollout on average, 31 min per run.